Hi! Choosing a movie is a real struggle for many of us :) So most of the streaming platforms have inbuild recommendation systems. These systems aim to predict user's interests and recommend items that they'll probably like. Throughout this notebook, we will try to use 2 clasterisation methods to build our own movie recommender.

We are going to use three following data sets:

    Netflix TV Shows and Movies
    
    HBO Max TV Shows and Movies
    
    Amazon Prime TV Shows and Movies

# Step-1: Import Required Library

In [2]:
import numpy as np
import pandas as pd 
import warnings 
warnings.filterwarnings('ignore')

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_samples,silhouette_score
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler

# Step-2: Data imports using Pandas

In [30]:
df_netflix=pd.read_csv('Netflix/titles.csv')
df_hbo=pd.read_csv('HBO/titles.csv')
df_amazon=pd.read_csv('Amazon/titles.csv')

df=pd.concat([df_netflix,df_hbo,df_amazon],axis=0)
df.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


# Step-3: Data Cleaning and Preprocessing

In [31]:
df.shape

(19015, 15)

In [32]:
df.columns

Index(['id', 'title', 'type', 'description', 'release_year',
       'age_certification', 'runtime', 'genres', 'production_countries',
       'seasons', 'imdb_id', 'imdb_score', 'imdb_votes', 'tmdb_popularity',
       'tmdb_score'],
      dtype='object')

In [33]:
df.isna().sum()

id                          0
title                       1
type                        0
description               149
release_year                0
age_certification       10314
runtime                     0
genres                      0
production_countries        0
seasons                 14796
imdb_id                  1396
imdb_score               1875
imdb_votes               1912
tmdb_popularity           671
tmdb_score               2661
dtype: int64

In [60]:
df_movies = df.drop_duplicates()

In [61]:
df_movies.shape

(18980, 15)

In [62]:
df_movies.drop(['description','age_certification'],axis=1,inplace=True)

In [63]:
df_movies.shape

(18980, 13)

In [64]:
df_movies['production_countries']

0             ['US']
1             ['US']
2             ['US']
3             ['GB']
4       ['GB', 'US']
            ...     
9866          ['US']
9867          ['US']
9868          ['IN']
9869              []
9870              []
Name: production_countries, Length: 18980, dtype: object

In [65]:
df_movies['production_countries']=df_movies['production_countries'].str.replace(r"\[",'',regex=True).str.replace(r"\'",'',regex=True).str.replace(r"\]",'',regex=True)

In [66]:
df_movies['production_countries']

0           US
1           US
2           US
3           GB
4       GB, US
         ...  
9866        US
9867        US
9868        IN
9869          
9870          
Name: production_countries, Length: 18980, dtype: object

In [67]:
df_movies['lead_prod_country']=df_movies['production_countries'].str.split(',').str[0]

In [68]:
df_movies['lead_prod_country']

0       US
1       US
2       US
3       GB
4       GB
        ..
9866    US
9867    US
9868    IN
9869      
9870      
Name: lead_prod_country, Length: 18980, dtype: object

In [69]:
df_movies['prod_countries_cnt'] = df_movies['production_countries'].str.split(',').str.len()

In [70]:
df_movies['prod_countries_cnt']

0       1
1       1
2       1
3       1
4       2
       ..
9866    1
9867    1
9868    1
9869    1
9870    1
Name: prod_countries_cnt, Length: 18980, dtype: int64

In [71]:
df_movies['lead_prod_country'] = df_movies['lead_prod_country'].replace('', np.nan)

In [72]:
df_movies['lead_prod_country']

0        US
1        US
2        US
3        GB
4        GB
       ... 
9866     US
9867     US
9868     IN
9869    NaN
9870    NaN
Name: lead_prod_country, Length: 18980, dtype: object

In [73]:
df_movies['genres']

0                                 ['documentation']
1                                ['drama', 'crime']
2       ['drama', 'action', 'thriller', 'european']
3                   ['fantasy', 'action', 'comedy']
4                                 ['war', 'action']
                           ...                     
9866                                      ['drama']
9867                                     ['comedy']
9868                                      ['crime']
9869                            ['family', 'drama']
9870                                      ['drama']
Name: genres, Length: 18980, dtype: object

In [74]:
df_movies['genres'] = df_movies['genres'].str.replace(r"\[", '', regex=True).str.replace(r"'", '', regex=True).str.replace(r"\]", '', regex=True)
df_movies['genres']

0                           documentation
1                            drama, crime
2       drama, action, thriller, european
3                 fantasy, action, comedy
4                             war, action
                      ...                
9866                                drama
9867                               comedy
9868                                crime
9869                        family, drama
9870                                drama
Name: genres, Length: 18980, dtype: object

In [75]:
df_movies['main_genre'] = df_movies['genres'].str.split(',').str[0]
df_movies['main_genre']

0       documentation
1               drama
2               drama
3             fantasy
4                 war
            ...      
9866            drama
9867           comedy
9868            crime
9869           family
9870            drama
Name: main_genre, Length: 18980, dtype: object

In [76]:
df_movies['main_genre'] = df_movies['main_genre'].replace('', np.nan)
df_movies['main_genre']

0       documentation
1               drama
2               drama
3             fantasy
4                 war
            ...      
9866            drama
9867           comedy
9868            crime
9869           family
9870            drama
Name: main_genre, Length: 18980, dtype: object

In [77]:
df_movies.drop(['genres', 'production_countries'], axis=1, inplace=True)

In [78]:
df_movies.shape

(18980, 14)

In [79]:
df_movies.columns

Index(['id', 'title', 'type', 'release_year', 'runtime', 'seasons', 'imdb_id',
       'imdb_score', 'imdb_votes', 'tmdb_popularity', 'tmdb_score',
       'lead_prod_country', 'prod_countries_cnt', 'main_genre'],
      dtype='object')

In [80]:
df_movies.head()

,id,title,type,release_year,runtime,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_country,prod_countries_cnt,main_genre
0,ts300399,Five Came Back: The Reference Films,SHOW,1945,51,1.0,NaN,NaN,NaN,0.600,NaN,US,1,documentation
1,tm84618,Taxi Driver,MOVIE,1976,114,NaN,tt0075314,8.2,808582.0,40.965,8.179,US,1,drama
2,tm154986,Deliverance,MOVIE,1972,109,NaN,tt0068473,7.7,107673.0,10.010,7.300,US,1,drama
3,tm127384,Monty Python and the Holy Grail,MOVIE,1975,91,NaN,tt0071853,8.2,534486.0,15.461,7.811,GB,1,fantasy
4,tm120801,The Dirty Dozen,MOVIE,1967,150,NaN,tt0061578,7.7,72662.0,20.398,7.600,GB,2,war


In [81]:
df_movies.isnull().sum()

id                        0
title                     1
type                      0
release_year              0
runtime                   0
seasons               14772
imdb_id                1394
imdb_score             1873
imdb_votes             1910
tmdb_popularity         670
tmdb_score             2656
lead_prod_country      1160
prod_countries_cnt        0
main_genre              321
dtype: int64

In [82]:
df_movies.dropna(inplace=True)

In [83]:
df_movies.set_index('title', inplace=True)

In [84]:
df_movies.drop(['id', 'imdb_id'], axis=1, inplace=True)

In [85]:
df_movies.shape

(3294, 11)

In [86]:
df_movies.head()

,type,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_country,prod_countries_cnt,main_genre
title,,,,,,,,,,,
Monty Python's Flying Circus,SHOW,1969,30,4.0,8.8,73424.0,17.617,8.306,GB,1,comedy
Seinfeld,SHOW,1989,24,9.0,8.9,308824.0,130.213,8.301,US,1,comedy
Knight Rider,SHOW,1982,51,4.0,6.9,34115.0,50.267,7.500,US,1,scifi
Thomas & Friends,SHOW,1984,10,24.0,6.5,5104.0,42.196,6.500,GB,1,animation
Saved by the Bell,SHOW,1989,23,5.0,7.1,35034.0,19.855,8.000,US,1,family


In [88]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3294 entries, Monty Python's Flying Circus to Qing Luo
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   type                3294 non-null   object 
 1   release_year        3294 non-null   int64  
 2   runtime             3294 non-null   int64  
 3   seasons             3294 non-null   float64
 4   imdb_score          3294 non-null   float64
 5   imdb_votes          3294 non-null   float64
 6   tmdb_popularity     3294 non-null   float64
 7   tmdb_score          3294 non-null   float64
 8   lead_prod_country   3294 non-null   object 
 9   prod_countries_cnt  3294 non-null   int64  
 10  main_genre          3294 non-null   object 
dtypes: float64(5), int64(3), object(3)
memory usage: 308.8+ KB


# Step-3: Encoding Categorical Features

In [92]:
dummies = pd.get_dummies(df_movies[['type', 'lead_prod_country', 'main_genre']], drop_first=True)

df_movies_dum = pd.concat([df_movies, dummies], axis=1)

df_movies_dum.drop(['type', 'lead_prod_country', 'main_genre'], axis=1, inplace=True)

In [93]:
df_movies_dum.shape

(3294, 81)

In [94]:
df_movies_dum.head()

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,prod_countries_cnt,lead_prod_country_AR,lead_prod_country_AT,...,main_genre_history,main_genre_horror,main_genre_music,main_genre_reality,main_genre_romance,main_genre_scifi,main_genre_sport,main_genre_thriller,main_genre_war,main_genre_western
title,,,,,,,,,,,,,,,,,,,,,
Monty Python's Flying Circus,1969,30,4.0,8.8,73424.0,17.617,8.306,1,False,False,...,False,False,False,False,False,False,False,False,False,False
Seinfeld,1989,24,9.0,8.9,308824.0,130.213,8.301,1,False,False,...,False,False,False,False,False,False,False,False,False,False
Knight Rider,1982,51,4.0,6.9,34115.0,50.267,7.500,1,False,False,...,False,False,False,False,False,True,False,False,False,False
Thomas & Friends,1984,10,24.0,6.5,5104.0,42.196,6.500,1,False,False,...,False,False,False,False,False,False,False,False,False,False
Saved by the Bell,1989,23,5.0,7.1,35034.0,19.855,8.000,1,False,False,...,False,False,False,False,False,False,False,False,False,False


# Step-4: Scaling(MinMaxScalar)

In [95]:
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_movies_dum)
df_scaled = pd.DataFrame(df_scaled, columns=df_movies_dum.columns)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [96]:
df_scaled

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,prod_countries_cnt,lead_prod_country_AR,lead_prod_country_AT,...,main_genre_history,main_genre_horror,main_genre_music,main_genre_reality,main_genre_romance,main_genre_scifi,main_genre_sport,main_genre_thriller,main_genre_war,main_genre_western
0,0.397727,0.168539,0.058824,0.9125,0.037009,0.007913,0.815870,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.625000,0.134831,0.156863,0.9250,0.155671,0.058490,0.815326,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.545455,0.286517,0.058824,0.6750,0.017194,0.022579,0.728261,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.568182,0.056180,0.450980,0.6250,0.002570,0.018954,0.619565,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.625000,0.129213,0.078431,0.7000,0.017658,0.008919,0.782609,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3289,1.000000,0.162921,0.000000,0.5875,0.000194,0.000629,0.782609,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3290,0.988636,0.258427,0.000000,0.7500,0.000021,0.008224,0.728261,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3291,0.988636,0.264045,0.000000,0.8250,0.000012,0.002811,0.673913,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3292,0.988636,0.146067,0.000000,0.2750,0.000490,0.000628,0.271739,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Step-5: DBSCAN

In [113]:
eps_array = [0.2, 0.5, 1]
min_samples_array = [5, 10, 30]

In [114]:
for eps in eps_array:
    for min_sample in min_samples_array:
        clusters=DBSCAN(eps=eps,min_samples=min_sample)
        dbscan_label=clusters.fit_predict(df_scaled)

        if len(set(dbscan_label)) == 2:
            continue 
        silhouette_avg = silhouette_score(df_scaled, dbscan_label) 

        print("For eps =", eps,
              "For min_samples =", min_sample,
              "Count clusters =", len(set(dbscan_label)),
              "The average silhouette_score is :", silhouette_avg)


For eps = 0.2 For min_samples = 5 Count clusters = 75 The average silhouette_score is : 0.4378840737098286
For eps = 0.2 For min_samples = 10 Count clusters = 37 The average silhouette_score is : 0.36601440046646755
For eps = 0.2 For min_samples = 30 Count clusters = 17 The average silhouette_score is : 0.23106054247198204
For eps = 0.5 For min_samples = 5 Count clusters = 91 The average silhouette_score is : 0.601956050174035
For eps = 0.5 For min_samples = 10 Count clusters = 56 The average silhouette_score is : 0.5303679432698051
For eps = 0.5 For min_samples = 30 Count clusters = 21 The average silhouette_score is : 0.36228604161700484
For eps = 1 For min_samples = 5 Count clusters = 93 The average silhouette_score is : 0.6091664186394288
For eps = 1 For min_samples = 10 Count clusters = 57 The average silhouette_score is : 0.5362809971937993
For eps = 1 For min_samples = 30 Count clusters = 22 The average silhouette_score is : 0.37121300388037515


# Step-6 DBSCAN With Best Hypterparameters (eps=1, minpnts=5)


In [115]:
dbscan_model=DBSCAN(eps=1,min_samples=5).fit(df_scaled)
print("For eps =", 1,
      "For min_samples =", 5,
      "Count clusters =", len(set(dbscan_model.labels_)),
      "The average silhouette_score is :", silhouette_score(df_scaled, dbscan_model.labels_))

For eps = 1 For min_samples = 5 Count clusters = 93 The average silhouette_score is : 0.6091664186394288


In [116]:
df_movies['dbscan_clusters'] = dbscan_model.labels_

In [117]:
df_movies

,type,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_country,prod_countries_cnt,main_genre,dbscan_clusters
title,,,,,,,,,,,,
Monty Python's Flying Circus,SHOW,1969,30,4.0,8.8,73424.0,17.617,8.306,GB,1,comedy,0
Seinfeld,SHOW,1989,24,9.0,8.9,308824.0,130.213,8.301,US,1,comedy,1
Knight Rider,SHOW,1982,51,4.0,6.9,34115.0,50.267,7.500,US,1,scifi,2
Thomas & Friends,SHOW,1984,10,24.0,6.5,5104.0,42.196,6.500,GB,1,animation,3
Saved by the Bell,SHOW,1989,23,5.0,7.1,35034.0,19.855,8.000,US,1,family,4
...,...,...,...,...,...,...,...,...,...,...,...,...
Putham Pudhu Kaalai: Vidiyaadha,SHOW,2022,29,1.0,6.2,389.0,1.400,8.000,IN,1,drama,54
Chivas: El Rebaño Sagrado,SHOW,2021,46,1.0,7.5,46.0,18.308,7.500,MX,1,sport,-1
Be Yourself,SHOW,2021,47,1.0,8.1,29.0,6.259,7.000,CN,1,drama,45


# Step 7: Movie Recommendation Function

In [118]:
import random

In [124]:
def recommend_movie(movie_name: str):
    movie_name = movie_name.lower()
    df_movies['name'] = df_movies.index.str.lower()
    movie = df_movies[df_movies['name'].str.contains(movie_name, na=False)]

    if not movie.empty:
        cluster = movie['dbscan_clusters'].values[0]
        cluster_movies = df_movies[df_movies['dbscan_clusters'] == cluster]
        if len(cluster_movies) >= 5:
            recommended_movies = random.sample(list(cluster_movies.index), 5)
        else:
            recommended_movies = list(cluster_movies.index)
        print('--- We can recommend you these movies ---')
        for m in recommended_movies:
            print(m)
    else:
        print('Movie not found in the database.')


In [125]:
s = input('Input movie name: ')

print("\n\n")
recommend_movie(s)

Input movie name:  titans





--- We can recommend you these movies ---
Arrow
From the Earth to the Moon
Green Lantern: The Animated Series
Transformers: Prime
LEGO City Adventures


In [126]:
df_movies.to_csv("clustered_movies.csv", index=False)